In [1]:
from datetime import date

import hisepy
import pandas as pd
import polars as pl
import scanpy as sc
import json
import os
import tarfile
import session_info
import shutil

In [2]:
if os.path.isdir('results'):
    shutil.rmtree('results')

In [56]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Get the DEG Formatting Pipeline from Github

In [3]:
pipeline_repo = 'https://github.com/aifimmunology/DEG-Viewer-Pipeline'
pipeline_sha = 'b4572bec0621ff7f27a1a794efe568dd663a99cf'
pipeline_local_path = 'deg_pipeline'

In [4]:
clone_command = f'git clone {pipeline_repo} deg_pipeline; cd {pipeline_local_path}; git reset --hard {pipeline_sha}'

In [5]:
os.system(clone_command)

Cloning into 'deg_pipeline'...


HEAD is now at b4572be remove pl.lit for cutoffs


0

## Get DEGs

In [6]:
deg_uuid = '3279167f-a71c-442b-a325-2f778b4b07a1'

In [7]:
deg_file = hisepy.cache_files([deg_uuid])[0]

2026-06-16 14:54:05,040 INFO [hisepy.logging:185] logging 599 134201612662592 Calling cache_files
2026-06-16 14:54:12,749 INFO [hisepy.logging:228] logging 599 134201612662592 Finished cache_files successfully (time_elapsed=4.311s)


In [8]:
deg_file

'/home/workspace/input/1918706177/cohorts/3279167f-a71c-442b-a325-2f778b4b07a1/gold-neptunium-aluminum/tcell-vrd_deg_2026-06-15.csv'

In [12]:
deg = pl.read_csv(deg_file)

In [13]:
deg.head()

Cell Type,Treatment,Timepoint,fg,bg,gene,n_sample,mean,logFC,nomP,adjP
str,str,str,str,str,str,i64,f64,f64,f64,f64
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""A1BG-AS1""",648,0.085197,-0.008604,0.7403701,0.999237
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAGAB""",648,0.29036,-0.054295,0.145599,0.991887
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAK1""",648,0.895006,0.014816,0.876254,0.999237
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAMDC""",648,0.2063,0.043135,0.160231,0.991887
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAMP""",648,0.068101,-0.010012,0.7332528,0.999237


In [37]:
deg = deg.rename({'gene': 'Gene'})
deg.write_csv('temp_deg.csv')

### Build Cell type colors .csv file for pipeline

In [14]:
unique_types = deg['Cell Type'].unique().to_list()
unique_types.sort()
print(len(unique_types))
print(unique_types)

6
['CD4 CM', 'CD4 EM', 'CD4 Naive', 'CD4 Treg', 'CD8 Memory', 'CD8 Naive']


In [20]:
type_colors_dict = {
    'Type Order': list(range(1,8)),
    'Cell Type': [
        'CD4 Naive',
        'CD4 CM',
        'CD4 EM',
        'CD4 Treg',
        'CD8 Naive',
        'CD8 Memory'
    ],
    'Type Color': [
        '#93A7D1',
        '#00AEEF',
        '#1C75BC',
        '#E57A93',
        '#8DC63F',
        '#009444'
    ]
}
type_colors_df = pl.DataFrame(type_colors_dict)
type_colors_df.write_csv('cell_type_colors.csv')

## Assemble configuration and run pipeline

Make a samplesheet

In [38]:
samplesheet = pd.DataFrame({
    'File': ['temp_deg.csv'],
    'Type': ['DESeq2']
})

Save samplesheet for use by snakemake

In [39]:
samplesheet_file = 'samplesheet.csv'

samplesheet.to_csv(samplesheet_file, index = False)

Make configuration

In [47]:
config = {
    'project': {
        'title': 'T cell responses to VRd treatments',
        'file': samplesheet_file,
        'covariates': ['Treatment', 'Cell Type', 'Timepoint'],
        'samplesheet': samplesheet_file,
    },
    'deseq2': {
        'adjp_column': 'adjP',
        'adjp_cutoff': 0.05,
        'es_column': 'logFC',
        'es_cutoff': 0,
        'cell_type_annotation': 'cell_type_colors.csv',
        'rename_columns': {
        }
    }
}

Save configuration for use by snakemake

In [48]:
config_file = 'config.json'

with open(config_file, "w") as outfile:
    json.dump(config, outfile, indent=4)

## Run pipeline

In [49]:
snakefile_path = f'{pipeline_local_path}/workflow/Snakefile'
conda_wrapper = 'conda run -p /home/workspace/environment/dashdev10 '

#### Dry Run

In [50]:
dryrun_command = f'snakemake --snakefile {snakefile_path} --configfile {config_file} --dry-run'

In [51]:
os.system(conda_wrapper + dryrun_command)

host: dash-vis-dev-0
Building DAG of jobs...
Job stats:
job              count
-------------  -------
format_deseq2        1
all                  1
total                2


[Tue Jun 16 16:08:35 2026]
rule format_deseq2:
    input: temp_deg.csv
    output: results/deg/meta.parquet, results/deg/summaries.parquet, results/deg/adjp.parquet, results/deg/log2fc.parquet, results/deg/means.parquet, results/deg/t-adjp.parquet, results/deg/t-log2fc.parquet
    jobid: 1
    reason: Params have changed since last execution: Union of exclusive params before and now across all output: before: 0.2,0.01 now: 0,0.05 
    resources: tmpdir=/tmp
[Tue Jun 16 16:08:35 2026]
rule all:
    input: results/deg/meta.parquet, results/deg/summaries.parquet, results/deg/adjp.parquet, results/deg/log2fc.parquet, results/deg/means.parquet, results/deg/t-adjp.parquet, results/deg/t-log2fc.parquet
    jobid: 0
    reason: Input files updated by another job: results/deg/log2fc.parquet, results/deg/adjp.parquet, results

0

#### Actual Run

In [52]:
pipeline_command = f'snakemake --snakefile {snakefile_path} --configfile {config_file} --cores 1'

In [53]:
os.system(conda_wrapper + pipeline_command)

Assuming unrestricted shared filesystem usage.
host: dash-vis-dev-0
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 1 (use --cores to define parallelism)
Rules claiming more threads will be scaled down.
Conda environments: ignored
Job stats:
job              count
-------------  -------
format_deseq2        1
all                  1
total                2

Select jobs to execute...
Execute 1 jobs...

[Tue Jun 16 16:08:40 2026]
localrule format_deseq2:
    input: temp_deg.csv
    output: results/deg/meta.parquet, results/deg/summaries.parquet, results/deg/adjp.parquet, results/deg/log2fc.parquet, results/deg/means.parquet, results/deg/t-adjp.parquet, results/deg/t-log2fc.parquet
    jobid: 1
    reason: Params have changed since last execution: Union of exclusive params before and now across all output: before: 0.01,0.2 now: 0.05,0 
    resources: tmpdir=/tmp
[Tue Jun 16 16:08:42 2026]
Finished jobid: 1 (Rule: format_deseq2)
1 of 2 steps (50%) done
Select jobs to execu

0

## Tar outputs and save to HISE

In [54]:
out_tar = 'tcell-vrd_deg_vis_{d}.tar'.format(d = date.today())
with tarfile.open(out_tar, "w") as tar:
    tar.add('results')

In [57]:
search_id = element_id()
search_id

'silver-copernicium-fluorine'

In [58]:
session_info.show()

In [60]:
hisepy.upload_files(
    files = [out_tar],
    study_space_id = '40df6403-29f0-4b45-ab7d-f46d420c422e',
    title = 'T cell VRd DEG files formatted for DEG Explorer {d}'.format(d = date.today()),
    input_file_ids = [deg_uuid],
    destination = search_id
)

2026-06-16 16:12:17,393 INFO [hisepy.logging:185] logging 599 134201612662592 Calling upload_files
2026-06-16 16:12:17,395 INFO [hisepy.logging:185] logging 599 134201612662592 Calling upload_files_internal


Please provide input of comma separated sample ids or sample kit guids for the files being uploaded. If you do not have any samples, press enter:  


2026-06-16 16:12:20,922 INFO [hisepy.logging:185] logging 599 134201612662592 Calling get_default_store
2026-06-16 16:12:24,125 INFO [hisepy.logging:228] logging 599 134201612662592 Finished get_default_store successfully (time_elapsed=0.441s)
2026-06-16 16:12:36,226 INFO [hisepy.logging:185] logging 599 134201612662592 Calling conda_env_builds
2026-06-16 16:12:36,227 INFO [hisepy.logging:53] utils 599 134201612662592 Starting conda environment build validation...
2026-06-16 16:12:36,229 INFO [hisepy.logging:75] utils 599 134201612662592 Exporting conda environment from /home/workspace/environment/dashdev10...
2026-06-16 16:12:38,901 INFO [hisepy.logging:88] utils 599 134201612662592 Removing hisepy references from exported environment file...
2026-06-16 16:12:38,907 INFO [hisepy.logging:99] utils 599 134201612662592 Creating temporary conda environment at /tmp/conda_env_test_7d6j60jf/env_bde7df77de4141799215b27462d14a36...
2026-06-16 16:14:52,809 INFO [hisepy.logging:119] utils 599 13

{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': 'b4c0d0f0-2d1b-4a3b-997b-0e00ad4d39e2',
 'ProcessId': '7add5b64-8953-43fc-920c-5a90c1d6eb9e',
 'WorkflowId': '567b2add-6f7a-4607-be13-b3a974a65b66',
 'FileIds': ['0e795da8-538a-474c-81ba-d4d45303de66']}